In [ ]:
import csv
import os

def get_first_avg_from_file(filepath):
    """
    读取文件，返回第一个出现的 'Avg' 后面的数值。
    """
    # 检查文件是否存在
    if not os.path.exists(filepath):
        return "File Not Found"
    
    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            for line in f:
                stripped_line = line.strip()
                # 找到以 Avg 开头的行
                if stripped_line.startswith("Avg"):
                    parts = stripped_line.split()
                    if len(parts) >= 2:
                        return parts[1]  # 返回数值并立即停止，只取第一个
        return "Avg Not Found"
    except Exception as e:
        return f"Error: {str(e)}"

def main():
    # 1. 配置路径
    instance_list_file = 'instance.txt'      # 假设 instance.txt 还在当前目录
    result_folder = 'UncertaintyDistributionn/BaseHigh'               # 结果文件所在的文件夹名
    output_csv_file = 'result_baseHigh.csv'   # 输出文件名
    
    # 2. 定义后缀
    suffixes = ["_sol_SimDet.txt", "_sol_SimEDD.txt", "_sol_SimMAG.txt"]
    
    # 3. 开始处理
    with open(output_csv_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        
        # 写入表头
        header = ['Instance Name', 'SimDet', 'SimEDD', 'SimMAG']
        writer.writerow(header)
        
        # 读取算例列表
        try:
            with open(instance_list_file, 'r', encoding='utf-8') as f:
                instances = [line.strip() for line in f if line.strip()]
        except FileNotFoundError:
            print(f"错误：找不到 {instance_list_file}，请确保它在脚本同级目录下。")
            return

        print(f"开始在文件夹 '{result_folder}' 中处理 {len(instances)} 个算例...")

        for instance_name in instances:
            row = [instance_name]
            
            for suffix in suffixes:
                # 拼接文件名：算例名 + 后缀
                file_name = instance_name + suffix
                
                # 拼接完整路径：Baseline文件夹 + 文件名
                full_path = os.path.join(result_folder, file_name)
                
                # 获取结果
                avg_value = get_first_avg_from_file(full_path)
                row.append(avg_value)
            
            writer.writerow(row)
            
    print(f"处理完成！结果已保存至 {output_csv_file}")

if __name__ == "__main__":
    main()

开始在文件夹 'Distribution/BaseHigh' 中处理 72 个算例...
处理完成！结果已保存至 result_baseHigh.csv


In [ ]:
##显著性统计分析：SimHGSvsMVND

In [1]:
import pandas as pd
from scipy import stats
import numpy as np
import warnings

def analyze_significance_all(file_path):
    print(f"正在读取文件: {file_path} ...")
    
    # -------------------------------------------------------
    # 1. 数据读取与清洗
    # -------------------------------------------------------
    try:
        # header=1: 跳过第一行大标题(SimHGS, MVND)，使用第二行(Best, Avg...)作为列名
        df = pd.read_csv(file_path, header=1)
        
        # 强制重命名列，确保万无一失
        # 对应顺序: Instances | SimHGS(Best) | SimHGS(Avg) | MVND(Best) | MVND(Avg)
        df.columns = ['Instances', 'SimHGS_Best', 'SimHGS_Avg', 'MVND_Best', 'MVND_Avg']
        
        # 清洗 Instances 列，去除可能存在的空格
        df['Instances'] = df['Instances'].astype(str).str.strip()
        
        # 提取后缀作为分组标识 (A, B, C)
        df['Set_ID'] = df['Instances'].apply(lambda x: x[-1])
        
    except Exception as e:
        print(f"读取错误: {e}")
        return

    # -------------------------------------------------------
    # 2. 定义分析组：包括 A, B, C 和 All (整体)
    # -------------------------------------------------------
    # 这里增加了 'All' 选项
    target_groups = ['A', 'B', 'C', 'All']
    
    comparisons = [
        ('Best', 'SimHGS_Best', 'MVND_Best'),
        ('Avg',  'SimHGS_Avg',  'MVND_Avg')
    ]

    # 打印表头
    print(f"\n{'='*75}")
    print(f"{'Group':<6} | {'Metric':<10} | {'Count':<5} | {'p-value':<10} | {'Result':<15}")
    print(f"{'-'*75}")

    # -------------------------------------------------------
    # 3. 循环分析
    # -------------------------------------------------------
    for group in target_groups:
        
        # 核心逻辑：如果是 'All'，取全部数据；否则取对应子集
        if group == 'All':
            subset = df
        else:
            subset = df[df['Set_ID'] == group]
        
        # 如果该组没有数据，跳过
        if subset.empty:
            continue
            
        count = len(subset) # 当前组的算例数量

        for metric_name, col1, col2 in comparisons:
            data_sim = subset[col1]
            data_mvnd = subset[col2]
            
            # 计算差值
            diff = data_sim - data_mvnd
            
            # --- 检验逻辑开始 ---
            if np.all(diff == 0):
                # 如果所有解都完全一样
                p_value = 1.0
                result = "Identical (=)"
            else:
                try:
                    # Wilcoxon 符号秩检验 (two-sided)
                    stat, p_value = stats.wilcoxon(data_sim, data_mvnd, zero_method='wilcox', correction=True)
                    
                    if p_value < 0.05:
                        result = "Significant *" # 显著差异
                    else:
                        result = "No Sig. Diff"  # 无显著差异
                except ValueError:
                    # 通常发生在样本量太小且非零差值不足以计算时
                    p_value = 1.0
                    result = "Error/Identical"
            # --- 检验逻辑结束 ---

            # 格式化输出
            print(f"{group:<6} | {metric_name:<10} | {count:<5} | {p_value:.4e}     | {result:<15}")
        
        # 每个组分析完后打印个分隔线（可选）
        if group != 'All':
            print(f"{'-'*75}")

    print(f"{'='*75}")
    print("注1: 'Count' 表示参与检验的算例数量")
    print("注2: p-value < 0.05 表示拒绝原假设(两个算法性能相同)，即存在显著差异")
    print("注3: Wilcoxon 检验要求数据成对出现")

if __name__ == "__main__":
    analyze_significance_all('Significance.csv')

正在读取文件: Significance.csv ...

Group  | Metric     | Count | p-value    | Result         
---------------------------------------------------------------------------
A      | Best       | 24    | 6.4115e-05     | Significant *  
A      | Avg        | 24    | 2.8888e-05     | Significant *  
---------------------------------------------------------------------------
B      | Best       | 24    | 1.1921e-07     | Significant *  
B      | Avg        | 24    | 1.1921e-07     | Significant *  
---------------------------------------------------------------------------
C      | Best       | 24    | 1.5268e-04     | Significant *  
C      | Avg        | 24    | 5.9605e-07     | Significant *  
---------------------------------------------------------------------------
All    | Best       | 72    | 5.8470e-12     | Significant *  
All    | Avg        | 72    | 5.1056e-13     | Significant *  
注1: 'Count' 表示参与检验的算例数量
注2: p-value < 0.05 表示拒绝原假设(两个算法性能相同)，即存在显著差异
注3: Wilcoxon 检验要求数据成对出现
